On DAP:
* unzip the project:
<br> ```unzip folder.zip```

* Delete folders on dap: 
<br>```mc rm --force --purge dap/corealgos/rmilocco/repo/test/```

On local:
* Zip outside the ``graph-ensembles`` folder: 
<br>```zip -rX graph-ensembles.zip graph-ensembles -x "graph-ensembles/src/graph_ensembles.egg-info/*" ".*" "*/.*" "*/__pycache__/*"```

In [1]:
# import graph_ensembles as ge
# import os, sys

# for rel_dir in ["src/graph_ensembles/", "src/"]:
#     newdir = os.path.join(os.path.dirname(os.getcwd()), rel_dir)
#     if newdir not in sys.path:
#         sys.path.insert(0, newdir)

# import graph_ensembles as ge

In [2]:
import graph_ensembles as ge

Remember that one can't install ipykernel in .venv. So, don't create a ```.venv``` if you want to use Jupyter;

To install graph-ensembles, go in the terminal (virtual-env) and type

* ```python3 -c "import graph_ensembles"``` (ensure there is no ```graph-ensembles``` package);
* From the project dir, ```pip install -e .``` (updates packages when changed) ;
    * If don't work, try with ```pip install install .``` (packages in src are freezed)
* ```python3 -c "import graph_ensembles"``` to check it was correctly installed;

In [2]:
try:
    corpkey = True if os.environ['DSBOX_USERNAME'] else None
    %pip install matplotlib pandas scipy scikit-learn numba
except:
    corpkey = None

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os, sys

# auto-reload the packages at every run
%load_ext autoreload
%autoreload 2

#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

for rel_dir in ["src/graph_ensembles/", "src/"]:
    newdir = os.path.join(os.path.dirname(os.getcwd()), rel_dir)
    if newdir not in sys.path:
        sys.path.insert(0, newdir)

# from Directed_SIM import *
# from Directed_Graph import Directed_Graph
# from directed_plotting_functions import *
# from param_rearranger import DiParam_Rearranger
import graph_ensembles as ge

dataset_name = "ING" #"recNET"
dataset_direction = "Directed"
id_code, cg_method, year = "grid_id", "random", 2022
distance_matrix, lvl_to_nclust = None, None

In [3]:
import graph_ensembles as ge

In [4]:
# note that inside the kwargs there is a copy of the pdtrans
pdtrans, kwargs, total_levels = \
    ge.dataset_loader(dataset_name, dataset_direction = dataset_direction,
                    corpkey = corpkey, id_code = id_code, cg_method = cg_method,
                    year = year, max_n_entries = 0)


Reading from local source


Define all the vertex and edges

In [5]:
pdf = pdtrans
e = pdf.loc[:, [f'payer_{id_code}', f'beneficiary_{id_code}']]
unique_nodes_from = lambda df: pd.DataFrame(data = pd.unique(df.to_numpy().ravel('K')), columns = ["id"])
v = unique_nodes_from(e) 
v_id = v.columns
e = pdf.iloc[:, ::2] # payer, beneficiary, amount_euro
e.columns = ["src", "dst", "amount"] 

In [6]:
e.head()

# check for self-loops
mask = e['src'] == e['dst']
print(f'-There are {mask.sum()} self-loops',)

,src,dst,amount
0,10000080,50842734,2706.50
1,10000087,36927785,126324.00
2,10000109,36110655,5615.20
3,10000109,38555565,3668.25
4,10000159,41790521,1306.80


-There are 0 self-loops


Create the full adjacency matrix

In [7]:
def load_or_create_degrees(g):
    """ 
    Calculate the degrees or Load them 
    param: g
    """
    from graph_ensembles.utils import load_array

    path_degree = lambda x: g.vars_dir + f"/{x}_degree.csv"
    prefix_exp = "_exp" if g.kind == "exp" else ""
    if os.path.exists(path_degree(f"{prefix_exp}_out")):
        # load the degrees
        print('-Load the degrees',)
  
        if g.kind == "obs":
            g._in_degree = load_array(path_degree("_in"))
            g._out_degree = load_array(path_degree("_out"))
            g._degree = load_array(path_degree(""))
        else:
            g._exp_in_degree = load_array(path_degree("_exp_in"))
            g._exp_out_degree = load_array(path_degree("_exp_out"))
            g._exp_degree = load_array(path_degree("_exp"))
    
    else: 
        print('-Calculate the degrees',)	
        if g.kind == "obs":
            _ = g.out_degree()
            _ = g.degree()
            
            np.savetxt(path_degree("_in"), g._in_degree, delimiter = ",")
            np.savetxt(path_degree("_out"), g._out_degree, delimiter = ",")
            np.savetxt(path_degree(""), g._degree, delimiter = ",")
        else:
            _ = g.expected_degree()
            np.savetxt(path_degree("_exp_in"), g._exp_in_degree, delimiter = ",")
            np.savetxt(path_degree("_exp_out"), g._exp_out_degree, delimiter = ",")
            np.savetxt(path_degree("_exp"), g._exp_degree, delimiter = ",")
        

import sparse as ge

kwargs_graph = {'name' : 'ING', 'level' : 0, 'corpkey' : corpkey}
g = ge.graphs.DiGraph(v, e, **kwargs_graph)

Create a split of ``v``

In [8]:
N = v.shape[0]
perc_ing_nodes = 0.7
num_ing_nodes = int(perc_ing_nodes * N)

print(f'-num_ing_nodes: {num_ing_nodes}',)

# fixed a seed, extract num_ing_nodes indexes for the vI nodes 
seed = 0
np.random.seed(seed)
idx_ing_nodes = np.random.choice(N, size = num_ing_nodes, replace=False)
vI = v.iloc[idx_ing_nodes].sort_values(by = "id", ignore_index = True)

-num_ing_nodes: 203886


### Scenario 1

1) Find ``e`` as the edges with both selected ``v``;
2) Refind ``v`` s.t. they have at least 1 link in ``e``;
3) Fit $\delta_{ING}$ via $\langle L_I \rangle$;
4) Predict $\langle L_R \rangle$ ROW (unseen) counterpart;

In [9]:
fit_method = "num_edges_intra"
if fit_method == "num_edges_intra":
    
    # find idx of edges containing v and filter edges
    idx_with_both_ = lambda v: e['src'].isin(v['id']) & e['dst'].isin(v['id'])
    edge_idx = lambda idx: e.loc[idx].reset_index(drop = True)
    
    # containing vI, vR
    idx_with_both_vI = idx_with_both_(vI)

    # select the edge ING
    eI = edge_idx(idx_with_both_vI)

    # re-obtain the vI nodes, since there may be an ING node connected only to ROW nodes
    # vI = pd.DataFrame(data = pd.unique(eI.iloc[:,:2].to_numpy().ravel('K')), columns = ["id"])
    vI = unique_nodes_from(eI.iloc[:,:2])

    # # select the rest-of-the-world vertex
    idx_v_row = np.setdiff1d(np.squeeze(v.values), np.squeeze(vI.values), assume_unique=True)
    vR = pd.DataFrame(data = idx_v_row, columns = ["id"])

    # select the edge ROW
    eR = edge_idx(~idx_with_both_vI)

    assert vI.shape[0] + vR.shape[0] == N, "Some nodes are not present either in the ING or ROW nodes"
    assert eI.shape[0] + eR.shape[0] == e.shape[0], "Some edges are not present either in the ING or ROW nodes"

In [10]:
# Create graph object for lowest level classification
import sparse as ge

kwargs_graph = {'name' : 'ING', 'perc_ing_nodes' : perc_ing_nodes, 
                               'seed' : seed, 'level' : 0,
                               'full_intra_row' : "intra",
                               }
gI = ge.graphs.DiGraph(vI, eI, **kwargs_graph)
load_or_create_degrees(gI)

-Load the degrees


In [11]:
# create another dictionary for gI
kwargs_model = kwargs_graph.copy()
kwargs_model.update({"name" : "invariant", "fit_method" : "fit_intra"})
model = ge.ScaleInvariantModel(gI, **kwargs_model)

In [18]:
# load the invariant model on the gI or fit it
path_param = model.vars_dir + "/param.csv"
if os.path.exists(path_param):
    from graph_ensembles.utils import load_array
    print(f'-Load',)
    # load the param
    model.param = np.expand_dims(load_array(path_param), axis = 0) # The code needs np.array([#])
    
else: 
    print('-Fit',)
    x0 = [1.8996372e-17]
    model.fit(x0 = x0[0], maxiter = 10, verbose = 2) # self.param inside the fit() function
    # save the param
    np.savetxt(path_param, model.param, delimiter = ",")

# calculate the degrees
load_or_create_degrees(model)

-Load
-Load the degrees


In [36]:
import plots.plotting_functions as ppf

In [38]:
ppf.plots_exp_degree(gI, model, model)

In [39]:
np.max(np.abs(model._exp_in_degree.sum() - gI._num_edges))
np.max(np.abs(model._exp_out_degree.sum() - gI._num_edges))

1.862645149230957e-09

1.3969838619232178e-09

In [ ]:
import graph_ensembles.sparse.models as models
i,j = 0,1
self = model
x_i, y_j = self.prop_out[i], self.prop_in[j]
z_ij = models.GraphEnsemble.prop_dyad(i,j)
model.p_ij(model.param, x_i, y_j, z_ij)

1.2946016268014524e-10

In [ ]:
self.exp_degrees(
    self.p_ij,
    self.param,
    self.prop_out,
    self.prop_in,
    models.GraphEnsemble.prop_dyad,
    self.num_vertices,
    self.selfloops,
    )

(array([0.90662116, 1.50746452, 0.11270769, ..., 2.36456278, 0.99105399,
        0.16902342], shape=(291267,)),
 array([0.03184673, 1.4782179 , 0.10919449, ..., 0.48187307, 0.9052014 ,
        0.05844003], shape=(291267,)),
 array([0.87496662, 0.02962807, 0.00351508, ..., 1.88919867, 0.08641928,
        0.11063059], shape=(291267,)))

In [ ]:
self.p_ij,
self.prop_out,
self.prop_in,
self.prop_dyad,
self.num_vertices,
self.selfloops,
# model.param = 
model.expected_num_edges()

(CPUDispatcher(<function ScaleInvariantModel.p_ij at 0x32b9971c0>),)

(array([  2706.5 , 126324.  ,   9283.45, ...,  41022.1 ,  77185.06,
          4966.96], shape=(291267,)),)

(array([ 74608.1 ,   2518.01,    298.72, ..., 161730.3 ,   7345.85,
          9404.77], shape=(291267,)),)

(<bound method GraphEnsemble.prop_dyad of <graph_ensembles.sparse.models.invariant.ScaleInvariantModel object at 0x32f233ac0>>,)

(291267,)

(False,)

TypeError: too many arguments: expected 6, got 7

### My Code

Function fun, jac

In [ ]:
# Not chunked version
from numba import njit, prange
@njit()
def p_jac_ij(d, x_i, y_j, z_ij):
    if (x_i == 0) or (y_j == 0) or (z_ij == 0):
        return 0.0, 0.0

    if d[0] == 0:
        return 0.0, x_i * y_j * z_ij

    tmp = x_i * y_j * z_ij
    tmp1 = d[0] * tmp
    if np.isinf(tmp1):
        return 1.0, 0.0
    else:
        return -np.expm1(-tmp1), tmp * np.exp(-tmp1)

@njit(parallel=True)
def exp_edges_f_jac(p_jac_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    N = len(prop_out)

    # Preallocate result vectors for each outer loop iteration (i)
    # These arrays store intermediate totals per i, which can be summed later
    f_vector = np.zeros(N)
    jac_vector = np.zeros(N)

    # Outer loop is parallelized with prange
    # This is the correct and efficient use of numba's parallelism
    for i in prange(N):
        p_out_i = prop_out[i]

        # Use scalar accumulators for better memory efficiency and cache usage
        f_i = 0.0
        jac_i = 0.0

        # Inner loop is serial — this is good because nested prange is not well supported
        for j in range(N):
            if (i != j) or selfloops:
                p_in_j = prop_in[j]

                # Call the user-defined function to get value and jacobian for dyad (i, j)
                p_val, jac_val = p_jac_ij(param, p_out_i, p_in_j, prop_dyad(i,j))

                # Accumulate results efficiently in scalars
                f_i += p_val
                jac_i += jac_val

        # Store per-node results
        f_vector[i] = f_i
        jac_vector[i] = jac_i

    # Sum across all nodes to get final result (parallel reduction is fast for large n)
    f_vector, jac_vector = np.sum(f_vector), np.sum(jac_vector)

    return f_vector, jac_vector

In [ ]:
self = model
f, jac = exp_edges_f_jac( \
            p_jac_ij, x0, self.prop_out, self.prop_in, self.prop_dyad, self.selfloops \
        )

f - self.num_edges

1383339698.53939

Chunked version (in principle 3seconds faster)

In [ ]:
@njit(parallel=True)
def chunk_sum(delta, out_chunk, stre_in, i_start):
    chunk_size = out_chunk.shape[0]
    N = stre_in.shape[0]
    single_chunk_f = np.zeros(chunk_size, dtype=np.float64)  # Changed to an array
    single_chunk_jac = np.zeros(chunk_size, dtype=np.float64)
    
    for i in prange(chunk_size):  # parallel over rows therefore sum p_ij, jac_ij inside the for-loop
        i_global = i_start + i
        for j in range(N):
            if i_global != j:
                p_ij, jac_ij = p_jac_ij(delta, out_chunk[i], stre_in[j], 1.0)
                single_chunk_f[i] += p_ij  # Accumulate into result array
                single_chunk_jac[i] += jac_ij
    return np.sum(single_chunk_f), single_chunk_jac.sum()  # Sum the array to a scalar.
    
@njit
def expL_chunked(delta, stre_out, stre_in, chunk_size=1000):
    N = len(stre_out)
    N_chunks = N // chunk_size
    
    chunk_f = np.zeros(N_chunks, dtype=np.float64)
    chunk_jac = np.zeros(N_chunks, dtype=np.float64)
    
    for i_start in prange(0, N, chunk_size): # parallel loop
        i_end = min(i_start + chunk_size, N)
        
        # select the strength out in the chunk
        out_chunk = stre_out[i_start:i_end]

        # calculate the chunk_sum
        fun, jac = chunk_sum(delta, out_chunk, stre_in, i_start)
        
        # calculate both the expL and the jacobian
        int_i = i_start // chunk_size
        chunk_f[int_i], chunk_jac[int_i] = fun, jac
    return chunk_f.sum(), chunk_jac.sum()

@njit
def residualL(delta, stre_out, stre_in, L):
    return expL_chunked(delta, stre_out, stre_in) - L

In [ ]:
expL_chunked(x0, net.stre_out, net.stre_in)

(1385719021.5393922, 1.742884947336626e+21)

Solver

In [ ]:
from graph_ensembles import solver

In [ ]:
# from solver import solver.monotonic_newton_solver
x0 = 1.49e-25
solver.monotonic_newton_solver(
                        x0 = np.array([x0]),
                        fun = lambda x: expL_chunked(x[0], net.stre_out, net.stre_in), #self.density_fit_layer(x[0], p_out, p_in),
                        target = net.n_edges,
                        # atol=atol,
                        # rtol=rtol,
                        x_l=0.0,
                        x_u=np.inf,
                        max_iter=10,
                        full_return=True,
                        verbose=True,
                    )

In [ ]:
def newton_raphson(x0, stre_out, stre_in, L, atol=1e-8, maxiter=3):
    """
    Newton-Raphson root finder using analytic Jacobian and positivity constraint on delta.
    """
    x = max(x0, 1e-20)  # Ensure initial guess is strictly positive

    for it in range(maxiter):
        f_x, df_dx = expL_chunked(x, stre_out, stre_in) #residualL(x, stre_out, stre_in, L)
        f_x += - L
        if abs(f_x) < atol:
            print(f'Converged at it {it}, x: {x}')
            return x

        if df_dx == 0.0:
            print(f'Zero derivative at iteration {it}, x: {x}')
            return x

        # Newton update
        x_new = x - f_x / df_dx

        # Enforce positivity constraint
        if x_new <= 0:
            x_new = x * 0.5  # Simple backtracking or damping

        print(f'it {it}, x = {x}, f_x = {f_x}, df_dx = {df_dx}, x_new = {x_new}')
        x = x_new
        
    print(f'Did not converge after {maxiter} iterations. Final x: {x}')
    return x

In [ ]:
# Usage example:
newton_raphson(2e-10, net.stre_out, net.stre_in, net.n_edges, maxiter=int(1e3))

In [ ]:
from solver import monotonic_newton_solver

fun = lambda x: expL_chunked(x, net.stre_out, net.stre_in)

x0 = 1e-10
monotonic_newton_solver(
    x0,
    fun,
    net.n_edges,
    atol=1e-24,
    rtol=1e-9,
    x_l=0,
    x_u=np.inf,
    max_iter=3,
    full_return=False,
    verbose=False,
)

In [ ]:
model_dims = [name+"MSBM" for name in ["degc"]] #"topw", "fitn", "stripe"]] #,  #list(range(1, 13)) + [15,20]}
model_dims = dict(zip(model_dims, [[1]]*len(model_dims)))
model_objective = "NetRec"
model_kwargs = {
                "name" : list(model_dims.keys())[0],
                "dimX" : 1,
                "initial_guess" : "random",
                "fc_direction" : "fg",
                "objective" : model_objective,
            }

# total_levels are the total_available_levels, but, starting from top_level, the total_levels = top_level + 1 
top_level = total_levels - 1
total_levels = top_level + 1
stripes_level = top_level + 1
print(f'-top_level, total_levels, stripes_level: {top_level, total_levels, stripes_level}',)

Full code

In [ ]:
for model_name in model_dims:
    list_rel_err = []

    print(f'-top_level, stripes_level: {top_level, stripes_level}',)
    
    for level in np.arange(top_level, -1, -1): #np.arange(top_level+1)[::-1]:
        kwargs.update({
                        "level" : level,
                        })
                        
        print(f'\n-Level {level} for {model_name} - dim {model_dims[model_name]}',)
        net = Directed_Graph(**kwargs)

        density = net.n_edges / (net.n_nodes * (net.n_nodes - 1))
        print(f'-net.n_nodes, net.n_edges, density: {net.n_nodes}, {net.n_edges}, {density:.4}',)

        # tail head edges
        tail_head_edges = net.pdtrans.iloc[:, :2].T.to_numpy() #// 10**net.level
        tail_head_edges = np.vstack((net.map_idcode2int(tail_head_edges[0]), 
                                    net.map_idcode2int(tail_head_edges[1])))
        net.rec_n_edges_deg(tail_head_edges, printf = False)

        # create the model for net
        add_model_kwargs = {
                            # to create the reduced model
                            "fc_nodes_out" : net.fc_nodes_out, "fd_nodes_out" : net.fd_nodes_out,
                            "fc_nodes_in" : net.fc_nodes_in, "fd_nodes_in" : net.fd_nodes_in,
                            "n_fcfd_nodes_out" : net.n_fcfd_nodes_out,
                            "n_fcfd_nodes_in" : net.n_fcfd_nodes_in,
                            "stripes_level" : stripes_level if prefix_in_(model_name) else None,
                            }
  
        # inherit the add_model_kwargs over the model_kwargs
        model_kwargs.update(add_model_kwargs)

        # play the game if the model is not complete		
        is_not_complete = net.n_nodes - net.fc_nodes.size > 0
        print(f'-n_total_fc_nodes: {net.fc_nodes.size}',)
        if is_not_complete:
            
            for dimX in model_dims[model_name]:

                model_kwargs.update({"dimX": dimX, "name" : model_name, "top_level" : top_level})
                DMSM = Directed_MSM(obs_net = net, **model_kwargs)

                
                # if not, fit it
                if not os.path.exists(DMSM.vars_dir + "/pmatrix.csv"):

                    if model_name.startswith("degcMSM"):
                        DMSM.deg_fit(net)
                
                    elif model_name.startswith(("stripe", "fitn", "topw")):
                        DMSM.n_edges_fit(net, initial_guess = "n_edges_fit")

                else:
                    print(f'-Loading the {DMSM.name} - {dimX}',)
                    DMSM.load_XY_pmatrix()

        if level == top_level - 1:
            break

    # 			# find the expected out/in degrees
    # 			DMSM.deg_in_out_11()

    # 		# create sum_model
    # 		model_kwargs.update({
    # 							"name" : f"fine-{DMSM.name}", "reduced_by" : False, 
    # 							"top_level" : top_level, "stripes_level" : add_model_kwargs["stripes_level"],
    # 							})
    # 		# model_kwargs.update({"stripes_level" : None if "stripe" not in model_kwargs["name"] else stripes_level})
    # 		sum_model = Directed_MSM(obs_net = net, **model_kwargs)

    # 		# if coarse-grained direction, sum the parameters
    # 		if DMSM.fc_direction == "cg":
    # 			sum_model.sum_XYw(obs_net = net, ref_model = DMSM)
            
    # 		# if not, create the fine model
    # 		else: 
    # 			path_pmatrix = sum_model.vars_dir + "/pmatrix.csv"
                
    # 			# if not existing --> create it
    # 			if not os.path.exists(path_pmatrix):

    # 				if level == sum_model.top_level:

    # 					# for topwDMSM, one does not have X and Y so do not copy them
    # 					if model_name.startswith("topw"):
    # 						sum_model.delta = DMSM.delta.copy()
    # 						sum_model.save_var(var = np.array([sum_model.delta]), var_name = "delta")
    # 					else:
    # 						sum_model.X = DMSM.X.copy()
    # 						sum_model.Y = DMSM.Y.copy()
    # 					sum_model.pmatrix = DMSM.pmatrix.copy()
                    
    # 				else:
                        
    # 					# check if stripe/fitn then enforce the n_of_links
    # 					if model_name.startswith("topw"):
    # 						print('-Fix fitted delta',)
    # 						sum_model.delta = load_array(full_path_retriever(sum_model, sum_model.top_level, meas = "delta", stripes_level=sum_model.stripes_level))

    # 						#create the unconditioned pmatrix
    # 						sum_model.pmatrix = -np.expm1(-sum_model.delta * sum_model.fractioned_w(net, top_level = sum_model.stripes_level))

    # 					else:
    # 						if prefix_in_(model_name, ["stripe", "fitn"]):
    # 							print('-Fix fitted delta',)
    # 							sum_model.delta = load_array(full_path_retriever(DMSM, sum_model.top_level, meas = "delta"))
                                
    # 							if "stripe" in sum_model.name:
    # 								net._set_stripe_out_in_vectors(sum_model.stripes_level)
    # 								sum_model.X, sum_model.Y = np.sqrt(sum_model.delta) * net.stripe_out, np.sqrt(sum_model.delta) * net.stripe_in
    # 							else:
    # 								sum_model.X, sum_model.Y = np.sqrt(sum_model.delta) * net.stre_out[:, None], np.sqrt(sum_model.delta) * net.stre_in[:, None]

    # 						else:
    # 							print('-Fractioning',)
    # 							# divide the top parameters
    # 							sum_model.frac_XYw(net)

    # 						#create the unconditioned pmatrix
    # 						sum_model.pmatrix = sum_model.pmatrix_gen(X = sum_model.X, Y = sum_model.Y)

    # 						# condition the pmatrix if the level is less than the top_levels
    # 						# sum_model.conditioned_pmatrix(net, DMSM)
                    
    # 				# save XY and pmatrix for all level <= top_level
    # 				sum_model.save_XY_pmatrix()
                
    # 			else:
    # 				if level < sum_model.top_level: print(f'-Loading the conditioned {sum_model.name} - {dimX}',)
    # 				else: print(f'-Loading the {sum_model.name} - {dimX}',)
                    
    # 				# load the XY, pmatrix and delta
    # 				sum_model.load_XY_pmatrix()
                                
    # 			# calculate the expected out/in degrees
    # 			sum_model.deg_in_out_11()
                
    # 			# relative error in the number of edges
    # 			rel_err_n_edges = (sum_model.n_edges - net.n_edges) / net.n_edges
    # 			print(f'-rel_err_n_edges: {rel_err_n_edges}',)
                
    # 			# track the relative error of the number of edges
    # 			sum_model.track_n_edges(list_rel_err, net)
   
    # 		if DMSM.fc_direction == "fg":
    # 			plots_deg_out_in_rec(net, DMSM, sum_model)
    # 			# plots_annad_IO(net, DMSM, sum_model)
    # 			if "stripe" not in sum_model.name and not model_name.startswith("topw"):
    # 				plots_XY_vs_stre_out_in(net, DMSM, sum_model)
                
    # 			if model_name.startswith("degc") and level < top_level:
    # 				plots_fracXY_per_sector(net, DMSM, sum_model)
        
    # 	if level == top_level:
    # 		break
    
    # # save the track n_edges after all the levels were exhausted
    # rel_err_path = os.path.dirname(sum_model.vars_dir)+f"/rel_err_n_edges_across_levels.csv"
    # if not os.path.exists(rel_err_path):
    # 	np.savetxt(rel_err_path, list_rel_err, delimiter = ",")

Avoiding ``for-loops`` and unpacking many functions to be reused

In [ ]:
import numpy.random as npr
ref_model, obs_net = DMSM, net
obs_net.top_level = DMSM.top_level
diff_bot_level = 1
N = DMSM.n_nodes

# Itop_2_ibot.values() is a list of lists, so bot_n_nodes is the max of maximum list (+1)
top_bin_adj = load_array(full_path_retriever(obs_net, level = obs_net.top_level, meas = "bin_adj"))

# the n of Delta parameters is the same as the active connections of a_IJ
top_edge_idx = np.where(top_bin_adj)
n_Delta_params = top_edge_idx[0].size

# all residuals: n_deg_out + n_deg_in + n_Delta_params
N = obs_net.n_nodes

# expand the delta matrix
# consider now 1 - e**(-delta x * y) = 1 - (e**(-delta))**(x*y) = 1 - g**(x*y) where 0 <= g <= 1 when 0 <= delta <= np.inf
# substitu zeros_like --> ones_like

squared_Delta = np.ones_like(top_bin_adj)

# load the mapping from Itop to ibot, to stretch the squared_Delta parameters
obs_net.Itop_2_ibot = obs_net.isource_2_itarget(obs_net, lsour = obs_net.top_level, ltar = obs_net.level)

# obs residual and indexes creation + compute row, col, value indexes
bot_edge_idx, stret_top_edge_idx = obs_net.obs_block_n_edges(top_edge_idx)

# initial conditions
meas_path = lambda meas: full_path_retriever(DMSM, level = DMSM.level, name = "degcDMSM", str_dimXBC = "dimX1", meas = meas).replace(f"top_level{DMSM.top_level}/", "")
load_ = lambda meas: load_array(meas_path(meas))
X, Y = load_("X"), load_("Y")

Delta_0 = obs_net.n_edges_IJ / np.mean(obs_net.n_edges_IJ) #npr.random(n_Delta_params) #obs_net.n_edges_IJ / np.mean(obs_net.n_edges_IJ)
X_Y_Delta_0 = np.concatenate((X, Y, Delta_0)) #npr.random(2*N) only for degrees

In [ ]:
from scipy.optimize import least_squares

# initial conditions
# if already optimized
X = np.genfromtxt(DMSM.vars_dir + "/X.csv", delimiter=",")
Y = np.genfromtxt(DMSM.vars_dir + "/Y.csv", delimiter=",")
Delta = np.genfromtxt(DMSM.vars_dir + "/Delta.csv", delimiter=",")
X_Y_Delta_0 = np.concatenate((X, Y, Delta))

# with delta
# bounds = (0, np.concatenate(([np.inf]*2*obs_net.n_nodes, [1]*n_Delta_params)))

# only degrees
# bounds = (0, np.concatenate(([1]*obs_net.n_nodes, [np.inf]*obs_net.n_nodes)))
bounds = (0, np.inf)
res = least_squares(lambda X_Y_Delta: DMSM.residuals(net, X_Y_Delta, squared_Delta, bot_edge_idx, top_edge_idx, stret_top_edge_idx), 
                        x0 = X_Y_Delta_0,
                        method = 'trf',
                        bounds = bounds,
                        verbose = 2,
                        max_nfev = 30)

In [ ]:
X = res.x[:N]
np.savetxt(DMSM.vars_dir + "/X.csv", X, delimiter=",")
Y = res.x[N:2*N]
np.savetxt(DMSM.vars_dir + "/Y.csv", Y, delimiter=",")
Delta = res.x[2*N:]
np.savetxt(DMSM.vars_dir + "/Delta.csv", Delta_IJ, delimiter=",")

In [ ]:
DMSM.deg_out = DMSM.zl_pmatrix.sum(1)
DMSM.deg_in = DMSM.zl_pmatrix.sum(0)
np.max(DMSM.deg_out - obs_net.deg_out)
np.max(DMSM.deg_in - obs_net.deg_in)

In [ ]:
# plot the degrees (out / in)
fig, axs = plt.subplots(1, 2, figsize = (20,7))
axis_scale = 'log'
obs_s, exp_s = 15, 15
axs[0].scatter(obs_net.deg_out,obs_net.deg_out, marker = 'o', color = 'b', s = obs_s, label = 'obs_net.deg_out')
axs[0].scatter(obs_net.deg_out,DMSM.deg_out, marker = 'x', color = 'r', s = exp_s, label = 'DMSM.deg_out')
axs[0].set(xscale = axis_scale, yscale = axis_scale, xlabel = 'obs_net.deg_out', ylabel = 'DMSM.deg_out',)
axs[0].legend()
axs[0].set_axisbelow(True)
axs[0].grid(True)

axs[1].scatter(obs_net.deg_in, obs_net.deg_in, marker = 'o', color = 'b', s = obs_s, label = 'obs_net.deg_in')
axs[1].scatter(obs_net.deg_in, DMSM.deg_in,  marker = 'x', color = 'r', s = exp_s, label = 'DMSM.deg_in')
axs[1].set(xscale = axis_scale, yscale = axis_scale, xlabel = 'obs_net.deg_in', ylabel = 'DMSM.deg_in',)
axs[1].legend()
axs[1].set_axisbelow(True)
axs[1].grid(True)


In [ ]:
from scipy.optimize import least_squares

# with delta
bounds = (0, np.concatenate(([np.inf]*2*obs_net.n_nodes, [1]*n_Delta_params)))

# only degrees
# bounds = (0, np.concatenate(([1]*obs_net.n_nodes, [np.inf]*obs_net.n_nodes)))
bounds = (0, np.inf)
res = least_squares(lambda X_Y_Delta: DMSM.residuals(net, X_Y_Delta, squared_Delta, bot_edge_idx, top_edge_idx, stret_top_edge_idx), 
                        x0 = X_Y_Delta_0,
                        method = 'trf',
                        bounds = bounds,
                        verbose = 2,
                        max_nfev = 1e2)

In [ ]:
np.max(DMSM.zl_pmatrix.sum(1) - obs_net.deg_out)

In [ ]:
def add_colors(dict_, colors = None):
    if colors == None:
        colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    
    for k,c in zip(list(dict_.keys()), colors):
        dict_[k].append(c)

In [ ]:
from directed_plotting_functions import plots_rel_err_n_edges_across_levels

if DMSM.get("objective").endswith("NetRec"):
    # model_names = ["fitnDMSM", "degcDMSM"]
    # colors from plt.rcParams['axes.prop_cycle'].by_key()['color']
    
    # name_mark_color = {'fitnDMSM' : ['*'], 'topw' : ['^'], 'CM' : ['<'], 'degcDMSM' : ['^'], 'maxlDMSM' : ['v'], 'stripeDMSM' : ['<']}
    name_mark_color = {'fitnDMSM' : ['*'], 'topwDMSM' : ['^'], 'degcDMSM' : ['>'], 'stripeDMSM' : ['<']}
    model_names = name_mark_color.keys()
    add_colors(name_mark_color)
    
    markers = [name_mark_color[x][0] for x in model_names]
    colors = [name_mark_color[x][1] for x in model_names]
    
    # refine the model names with prefix 
    model_names = ["fine-" + x for x in model_names]
    dims = [1] * len(model_names)

else:
    model_names = ["maxlMSM", "maxlMSM", "maxlMSM", "maxlMSM", ]
    dims = [1, 2, 8, 16,]
    markers = ['o', '^', 'o', '*', 'P', '^',]

# for multi-level adjacency matrix to check for modularity
# fig, ax = plt.subplots(1, total_levels, figsize = (24,7))

# for multi-level parameters sum VS refittied
# fig, ax2 = plt.subplots(figsize = (24,7))

for level in np.arange(total_levels):
    print(f'-Level: {level}',)
    kwargs.update({
            "level" : level,
            })

    net = Directed_Graph(**kwargs)

    # plot the adjacency matrix
    # ax[level].imshow(net.zl_bin_adj, cmap = 'viridis', interpolation = 'none')
    # ax[level].grid(False)
    # ax[level].set(title = f"Level {level}")

    # plot the sum VS refitted parameters
    # ax2.scatter()

    # any(net.deg_out - net.n_nodes - 1 == 0)
    # any(net.deg_in - net.n_nodes - 1 == 0)

    # if net.n_nodes - net.fc_nodes.size > 0 and ~np.any(net.deg == 0):
    # 	plot_triangles_at_c(
    # 						obs_net = net, ref_model = DMSM,
    # 						models = model_names, 
    # 						dims = 	dims,
    # 						markers = markers,
    # 						n_points = int(2e1)
    # 						)

    #	# in this plot use also fitnMSM
    # 	if 'fitnMSM' in model_names:
    # 		level_exoX_vs_topoX(obs_net = net, sum_model = fit_model,
    # 							fc_direction = "cg", title = "topox", sum_model_ms = 5)

# plots_rel_err_n_edges_across_levels(sum_model, model_names, total_levels, markers, colors, stripes_level = stripes_level)
plot_sumXY_vs_topXY(DMSM, net, "degcDMSM", top_level)

### Miscellanea

Check if the arrays in the ``residuals`` are the same

In [ ]:
# check if the two arrays of the np.where are the same
def check_all_equal(a, b):
    all_equal = lambda i: np.all(np.where(a)[i] == np.where(b)[i])
    return np.all(all_equal(0) == all_equal(1))

# check if the squared_Delta values are the unique one in the block of expanded squared_Delta
flag = 0
for I,J in zip(*np.where(squared_Delta)):
    i, j = obs_net.Itop_2_ibot[I], obs_net.Itop_2_ibot[J]
    unique_in_stretched = np.unique(DMSM.stretched_top_mat(obs_net, squared_Delta, bot_edge_idx, stret_top_edge_idx)[np.ix_(i,j)])
    if not (squared_Delta[I,J] == unique_in_stretched):
        flag = 1
        
        break
    
if flag == 0: print('-They are the same',)
else: print(f'-{I,J} --> {squared_Delta[I,J], unique_in_stretched} are not the same',)

print(f'-check_where_equal(squared_Delta, top_bin_adj): {check_where_equal(squared_Delta, top_bin_adj)}',)

Create the block_n_edges and dictionaries to same the ``I, J`` and ``Itop_2ibot[I], Itop_2_ibot[J]``

In [ ]:
def block_n_edges(obs_net, bot_pmatrix, bot_level):
    """ 
    Returns two dictionaries:
    1) IJ_2_ij with keys IJ and values the internal indexes i,j, e.g. '(0, 0)': [[0,1,2,3], [0,1,2,3]]
    2) IJ_2_n_edges with keys IJ and values the number of edges in the block, e.g. '(0, 0)': 18
    The blocks are the ones induced from the obs_net to delta_bot_level, whereas indexes are the bot_level ones.
    
    Note: It is supposed to be use only at the fitting level, i.e. obs_net.level == top_level
    """

    from pickle import dump, load
    from utils import load_array
    
    # load the bin_adjacency matrix of the bot_level (the one needed for inter/intra n_edges)
    bot_dir = full_path_retriever(obs_net, level = bot_level, meas = None) 
    bot_bin_adj = load_array(bot_dir + "/bin_adj.csv")
    
    # create the directory to save the dictionaries
    bot_dir_dict = bot_dir + f"/block_n_edges/from_level{obs_net.level}"
    os.makedirs(bot_dir_dict, exist_ok = True)
    full_path = f"{bot_dir_dict}/IJ_2_ij.pkl"

    if not os.path.exists(full_path):
        IJ_2_ij = {}
        IJ_2_n_edges = {}
        
        # load the mapping among upper indexes and lower ones
        Itop_2_ibot = obs_net.isource_2_itarget(obs_net, lsour = obs_net.level, ltar = bot_level)
        
        # shuffle over existing edges on the top_adj (net.bin_adj)
        for (I,J) in zip(*np.where(obs_net.bin_adj)):

            # select the i,j indexes of the bot_level
            i,j = Itop_2_ibot[I], Itop_2_ibot[J]
            
            # create the numpy indexes to filter out the block_adjacency matrix
            ix_ = np.ix_(i,j)

            # calculate the sum of the microscopic edges in the block
            n_edges_IJ = int(np.sum(bot_bin_adj[ix_]))

            bot_n_edges_IJ = np.sum(bot_pmatrix[ix_])

            # update the dictionary. Note that the keys will have a space, i.e. '(0, 0)'
            IJ_2_ij.update({f"{I,J}" : [i,j]})
            IJ_2_n_edges.update({f"{I,J}" : n_edges_IJ})

            # print(f'-I,J: {I,J}',)
            # print(f'-i,j: {i,j}',)
            # print(f'-bot_bin_adj[i,j]: {bot_bin_adj[i,j]}',)
            # print(f'-n_edges_IJ: {n_edges_IJ}',)
            # print(f'-IJ_2_ij: {IJ_2_ij}',)
        
        with open(full_path, "wb") as f:
            dump(IJ_2_ij, f)
        
        with open(f"{bot_dir_dict}/IJ_2_n_edges.pkl", "wb") as f:
            dump(IJ_2_n_edges, f)
        
    else:
        with open(full_path, "rb") as f:
            IJ_2_ij = load(f)
        with open(f"{bot_dir_dict}/IJ_2_n_edges.pkl", "rb") as f:
            IJ_2_n_edges = load(f)

    return IJ_2_ij, IJ_2_n_edges

Check ``summed X, summed Y`` from the finest level re-sum to the ``sum_model.X, sum_model.Y``

In [ ]:
bottom_dir = lambda meas: full_path_retriever(ref_model = sum_model, level = 0, meas = meas)
bot_X = load_array(bottom_dir("X.csv"))
bot_Y = load_array(bottom_dir("Y.csv"))

sum_ = lambda X: [X[net.mic2mac_int == c].sum() for c in np.unique(net.mic2mac_int)]
bot2top_X = sum_(bot_X)
bot2top_Y = sum_(bot_Y)

np.max(np.abs(bot2top_X - sum_model.X))
np.max(np.abs(bot2top_Y - sum_model.Y))

Check if ``zl_pmatrix`` and explicitly computed are the same

In [ ]:
np.all(sum_model.zl_pmatrix == -sum_model.frmv_diag(np.expm1(-sum_model.X[:, None] @ sum_model.Y[None, :])))

Check that ``w_IJ`` (obtained via pdtrans filtering naics and summing amounts) is the sum of the members

In [ ]:
# check if s^{in}_{IJ} = w_IJ = sum_{i \in I} sum_{j \in J} w_{ij}
lower_level_weights = np.genfromtxt(net.vars_dir.replace(f"level{level}", f"level0") + "/wei_adj.csv", delimiter = ",")

for I in range(net.n_nodes):

    where_0 = lambda arr: np.where(arr)[0]
    I_conn_J = where_0(net.wei_adj[I])

    for J in I_conn_J:
        # print(f'-I_conn_J: {I_conn_J}',)
        # J = I_conn_J[idx_col]
        # net.wei_adj[I, J]

        node_in_row = where_0(net.mic2mac_int == I)
        node_in_col = where_0(net.mic2mac_int == J)

        llw = lower_level_weights[np.ix_(node_in_row, node_in_col)]

        re = rel_err(np.sum(llw), net.wei_adj[I, J])
        if re > 1e-3:
            print('-RelErr Out of Threshold',)
            print(f'-I,J: {I,J}',)
            print(f'-node_in_row: {node_in_row}',)
            print(f'-node_in_col: {node_in_col}',)
            print(f'-llw:\n {llw}',)
            break

Check if the ``stripes_out`` and ``stripes_in`` have the non-zeros entries for the active (top) sectors

In [ ]:
prolem_flag = 0
for i in range(net.n_nodes):
    where_out_i = np.nonzero(net.stripe_out[i])[0]
    where_in_i = np.nonzero(net.stripe_in[i])[0]
    
    # if all(where_nonzero_in_i == 0):

    # select the naics payers, beneficiariesj
    naics_i = net.int2idcode[i]
    pay2naics_i = net.pdtrans.loc[:, "beneficiary_naics_code"] == naics_i

    # find the top payers that are sending to naics_i
    power_diff = 10 ** (stripes_level - net.level)
    top_payers_of_i = net.pdtrans[pay2naics_i].loc[:, "payer_naics_code"] // power_diff
    top_payers_of_i = top_payers_of_i.unique()

    # find the mapping from stripe_level naics to indexes
    top_unique_naics = np.unique(net.int2idcode // power_diff)
    top_idcode2node = dict(zip(top_unique_naics, range(len(top_unique_naics))))

    # map the payers found before into top indexes
    # they should be equal to the nonzero columns in net.stripe_in
    top_pay2i_idx = [top_idcode2node[naics] for naics in top_payers_of_i]
    top_i_idx = top_idcode2node[naics_i // power_diff]

    if all(net.stripe_out[i] == 0):
        out_idx_equal = True
    else: out_idx_equal = all(where_out_i == top_i_idx)
        
    if all(net.stripe_in[i] == 0):
        in_idx_equal = True
    else:
        in_idx_equal = all(where_in_i == top_pay2i_idx)
    

    if not (out_idx_equal and in_idx_equal):
        print(f'-Problems for node {i, out_idx_equal, in_idx_equal}',)
        print(f'-where_out_i, top_i_idx: {where_out_i, top_i_idx}',)
        print(f'-where_in_i, top_i_idx: \n{where_in_i}, \n{np.array(top_pay2i_idx)}',)
        prolem_flag = 1
    
    # if i == 20: break

if not prolem_flag:
    print('-All matched',)

Check stripes have the same values of ``stre_out, _in`` if summed over the ``axis = 1``

In [ ]:
# check stripe_out sums to stre_out
all(np.sum(net.stripe_out, 1) == net.stre_out)

# the max-abs error is 4e-5 but the magnitudes are important
# therefore, check for the relative error where stre_in != 0
rel_err = lambda x, y: np.abs(x - y) / np.where(y != 0, y, 1)
stripein_2_stre_in = np.sum(net.stripe_in, 1)
np.max(rel_err(stripein_2_stre_in, net.stre_in))